# Decoupled Producer-Consumer Latency & Throughput Profiler (Google Colab)

This notebook runs the **Decoupled Producer-Consumer Architecture** for the AWS Indian Judgments pipeline in Google Colab.

### Architectural Strategy:
- **Dedicated Download Producer Thread**: Continuously streams PDF downloads from S3, fully saturating available network bandwidth without pausing for CPU text extraction.
- **Bounded Queue Safety Ceiling (`maxsize=8`)**: Maintains at most **8 unprocessed PDFs** on local VM disk/RAM at any moment. If CPU workers lag behind, the downloader automatically pauses to prevent disk/memory pressure.
- **Parallel CPU Consumer Pool**: Background CPU workers execute text extraction (PyMuPDF), clean text, and entity extraction in parallel.
- **Immediate File Purge**: Raw PDFs are deleted from disk immediately after text extraction.
- **0% Google Drive IO Pollution**: Intermediate files are processed on local scratch (`/tmp/colab_scratch`). Only `checkpoint.json` and final Parquet files are stored on Google Drive.

## Step 1: Install Dependencies

In [ ]:
!pip install -q pymupdf pyarrow duckdb rich spacy requests urllib3 matplotlib psutil

## Step 2: Auto-Load Pipeline Code & Modules

In [ ]:
import os, sys
repo_dir = "/content/aws_indian_judgements"
if os.path.exists("/content") and not os.path.exists(repo_dir):
    print("Cloning repository into Colab environment...")
    !git clone https://github.com/duttadev/aws_indian_judgements.git {repo_dir}
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    os.chdir(repo_dir)
elif os.path.exists(repo_dir):
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    os.chdir(repo_dir)

print(f"Working Directory: {os.getcwd()}")
from config import PipelineConfig
from pipeline.storage import StorageManager
from batch_runner import BatchScheduler
from pipeline.decoupled_runner import DecoupledPipelineScheduler
print("✓ Producer-Consumer pipeline modules loaded successfully!")

## Step 3: Fetch Real S3 Judgment Keys

In [ ]:
import time, json, glob, requests
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt

def get_real_s3_keys(max_keys=100):
    for kfile in ["./data/hc_keys_sample.json", "./data/sc_keys_sample.json"]:
        if os.path.exists(kfile):
            with open(kfile, "r") as f:
                return json.load(f)[:max_keys]
    try:
        r = requests.get("https://indian-high-court-judgments.s3.amazonaws.com/?max-keys=100", timeout=10)
        root = ET.fromstring(r.text)
        ns = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
        return [elem.find("s3:Key", ns).text for elem in root.findall("s3:Contents", ns) if elem.find("s3:Key", ns).text.endswith(".pdf")][:max_keys]
    except Exception as e:
        print(f"Error fetching keys: {e}")
        return []

benchmark_keys = get_real_s3_keys(max_keys=100)
print(f"Loaded {len(benchmark_keys)} real S3 PDF benchmark keys.")

## Step 4: Run Producer-Consumer Benchmark Suite

In [ ]:
consumer_worker_options = [2, 4, 8, 12]
results = []

for workers in consumer_worker_options:
    print(f"\n==================================================")
    print(f" Testing Producer-Consumer Architecture:")
    print(f" • Producer: Continuous S3 Streaming Downloader")
    print(f" • Download Queue Ceiling: 8 PDFs max in queue")
    print(f" • Consumer CPU Threads: {workers}")
    print(f"==================================================")
    
    scratch_path = f"/tmp/benchmark_pc_workers_{workers}"
    os.makedirs(scratch_path, exist_ok=True)
    
    config = PipelineConfig(
        run_id=f"pc-bench-{workers}",
        base_output_dir=scratch_path,
        local_scratch_dir=scratch_path,
        s3_base_url="https://indian-high-court-judgments.s3.amazonaws.com",
        batch_size=50,
        max_workers=workers,
        keep_pdf_files=False,
        keep_intermediate_artifacts=True,
        resume_enabled=False
    )
    
    storage = StorageManager(config)
    scheduler = DecoupledPipelineScheduler(config, storage, max_queue_size=8)
    
    t0 = time.time()
    summary = scheduler.run_decoupled_pipeline(benchmark_keys)
    elapsed = time.time() - t0
    
    results.append({
        "cpu_consumers": workers,
        "elapsed_sec": round(elapsed, 2),
        "pdfs_per_sec": summary.get("throughput_pdfs_per_sec", 0),
        "pages_per_sec": summary.get("throughput_pages_per_sec", 0),
        "peak_ram_mb": summary.get("peak_ram_mb", 0)
    })

df_pc = pd.DataFrame(results)
print("\n=== Producer-Consumer Benchmark Results ===")
print(df_pc.to_string(index=False))

## Step 5: Visualize Streaming Throughput vs Consumer Threads

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(df_pc["cpu_consumers"], df_pc["pdfs_per_sec"], marker="o", color="#2ca02c", linewidth=2.5)
plt.title("Producer-Consumer Streaming Throughput (PDFs / sec)", fontweight="bold")
plt.xlabel("CPU Consumer Threads")
plt.ylabel("PDFs / sec")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()